# Download, host, and query your own AI model

This notebook demonstrates the complete local-model workflow:

1. Download a model and tokenizer
2. Load them into your notebook process
3. Query the hosted model through an interactive prompt

In [1]:
!pip install --quiet transformers ipywidgets torch accelerate pathlib

In [2]:
from pathlib import Path

MODELS_DIR = Path.home() / "ai_models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Models will be stored in: {MODELS_DIR}")

Models will be stored in: /home/jovyan/ai_models


## Load the Base LLM

In [3]:
# Check for GPU (cuda) or CPU
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    print(
        "Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024**3),
            1
        ),
        "GB"
    )

CUDA: False


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Beginner model: small enough for CPU, useful for a first local-model demo.
model_name = "google/flan-t5-base"

# Smaller CPU-only option:
# model_name = "google/flan-t5-small"

print(f"Downloading/loading model: {model_name}")
print(f"Cache directory: {MODELS_DIR}")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=MODELS_DIR,
)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    cache_dir=MODELS_DIR,
).to(DEVICE)

print("Model is hosted in this notebook process and ready to query.")

Downloading/loading model: google/flan-t5-base
Cache directory: /home/jovyan/ai_models


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model is hosted in this notebook process and ready to query.


In [5]:
# Confirm device running model - GPU/CUDA or CPU
print(f"Model running on : {next(model.parameters()).device}")

Model running on : cpu


## Query the hosted model

In [6]:
questions = [
    "What are the main components of the Solar System?",
    "How does the Internet work at a high level?",
    "What are some applications of artificial intelligence?",
    "Explain what makes a large language model useful.",
]

In [7]:
def ask(model, tokenizer, question, max_new_tokens=200):
    """Send a prompt directly to the locally hosted model."""
    inputs = tokenizer(question, return_tensors="pt").to(DEVICE)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        no_repeat_ngram_size=2,
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("Q:", question)
    print("A:", answer)
    print("-" * 80)
    return answer

## Run sample questions

The questions below let us verify that the downloaded model is loaded and generating answers. Total runtime is shown at the end.

In [8]:
%%time

for question in questions:
    ask(model, tokenizer, question)

Q: What are the main components of the Solar System?
A: atoms
--------------------------------------------------------------------------------
Q: How does the Internet work at a high level?
A: The Internet works at a high level by transferring data from one computer to another computer.
--------------------------------------------------------------------------------
Q: What are some applications of artificial intelligence?
A: Artificial Intelligence
--------------------------------------------------------------------------------
Q: Explain what makes a large language model useful.
A: A large language model can be used to describe a wide range of languages.
--------------------------------------------------------------------------------
CPU times: user 16.8 s, sys: 752 ms, total: 17.6 s
Wall time: 20.9 s


In [9]:
# Interactive prompt
# Enter a question in the text box and the locally hosted model will respond.
import ipywidgets as widgets
from IPython.display import display

question_box = widgets.Text(
    placeholder="Ask the local model a question...",
    description="Question:",
    layout=widgets.Layout(width="80%"),
)
ask_button = widgets.Button(description="Ask model", button_style="primary")
output = widgets.Output()

def submit_question(_):
    question = question_box.value.strip()
    if not question:
        return

    with output:
        output.clear_output(wait=True)
        ask(model, tokenizer, question, max_new_tokens=300)

ask_button.on_click(submit_question)
question_box.on_submit(submit_question)

display(question_box, ask_button, output)

print("Tip: press Enter or click 'Ask model' to submit a question.")

/tmp/ipykernel_6233/288311771.py:24: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  question_box.on_submit(submit_question)


Text(value='', description='Question:', layout=Layout(width='80%'), placeholder='Ask the local model a questio…

Button(button_style='primary', description='Ask model', style=ButtonStyle())

Output()

Tip: press Enter or click 'Ask model' to submit a question.
